# Notebook 06 — Results and Predictions

## Purpose

Refit the model selected for each task on all 1,199 labeled comments, then assign a predicted label to all 5,292 comments. The held-out test scores from notebook 05 stay the measure of performance. This notebook's agreement with the manual labels is calculated after the model has already seen those labels, so it is reported separately and is not a test result.

The comment comparison uses three samples of 10, drawn without replacement from different pools:

- 10 annotated comments, with the manual label and the refit prediction
- 10 unlabeled raw comments, with the refit prediction
- 10 further comments from the rows that remain, with the refit prediction

Phone-like strings in excerpts are replaced with `[PHONE REDACTED]`.

Predicted labels are model output. They are not corrected ground truth.


## 1. Load the selected models and the full corpus

Keep the held-out test scores separate from the prediction pass.


In [1]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

candidates = [Path.cwd(), Path.cwd().parent, Path(r"D:\NIRA_CASE_STUDY")]
ROOT = None
for candidate in candidates:
    if (candidate / "NIRA_Facebook_Datasets_5292.csv").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    raise FileNotFoundError("Project root not found.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.supervised import (
    EMBEDDING_MODEL_NAME,
    make_classical_pipeline,
    make_embedding_estimator,
)
from config.project_config import (
    CLASSIFICATION_TARGETS,
    EXPECTED_ANNOTATION_N,
    EXPECTED_RAW_N,
    RANDOM_SEED,
)

PREP = ROOT / "outputs" / "01_data_preparation" / "20261018_content_review"
TOPIC = ROOT / "outputs" / "04_topic_modeling" / "20261019_topic_models"
MODEL = ROOT / "outputs" / "05_model_development" / "20261019_supervised"
RUN_ID = "20261019_predictions"
OUT = ROOT / "outputs" / "06_results_predictions" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
PHONE_LIKE = re.compile(
    r"(?i)(?:\+?252[\s\-]?\d[\d\s\-]{6,}\d|\btell\s*:\s*\+?\d[\d\s\-]+|\b\d{10,15}\b)"
)

raw = pd.read_csv(PREP / "raw_corpus_working.csv").sort_values("source_row").reset_index(drop=True)
ann = pd.read_csv(PREP / "annotation_linked.csv")
embeddings = np.load(TOPIC / "comment_embeddings.npy")
selected = json.loads((MODEL / "selected_models.json").read_text(encoding="utf-8"))
topics = pd.read_csv(TOPIC / "topic_assignments.csv")
if len(raw) != EXPECTED_RAW_N or len(ann) != EXPECTED_ANNOTATION_N:
    raise AssertionError("Unexpected corpus or annotation size.")
if embeddings.shape[0] != len(raw):
    raise AssertionError("Embedding rows do not match the working corpus.")

raw["model_text"] = raw["comment_whitespace_normalized"].fillna("").astype(str)
raw["emb_row"] = np.arange(len(raw))
ann = ann.merge(
    raw[["record_key", "model_text", "emb_row", "post_id", "comment_original", "phone_like_flag"]],
    on="record_key",
    how="left",
    suffixes=("", "_raw"),
)
if ann["model_text"].isna().any():
    raise AssertionError("Annotation rows did not join to model text.")
print("Selected models:")
for task in CLASSIFICATION_TARGETS:
    print(task, selected["tasks"][task]["model"])
print("Output directory:", OUT)


Selected models:
policy_relevance complement_nb
feedback_type logistic_regression
topic complement_nb
sentiment complement_nb
Output directory: D:\NIRA_CASE_STUDY\outputs\06_results_predictions\20261019_predictions


### Interpretation

The winners loaded here are Complement naive Bayes for three tasks and logistic regression for `feedback_type`. The prediction pass uses those winners only. The other three candidates from notebook 05 are not applied to the corpus.


## 2. Refit the selected model for each task and predict all 5,292 comments

Train on the 1,199 labeled rows, then assign a predicted label to every comment.


In [2]:
predictions = raw[["source_row", "record_key", "post_id", "is_annotated"]].copy()
distribution_rows = []
agreement_rows = []

for task in CLASSIFICATION_TARGETS:
    model_name = selected["tasks"][task]["model"]
    labels = selected["tasks"][task]["classes"]
    y_labeled = ann[task].astype(str)
    if model_name == EMBEDDING_MODEL_NAME:
        scaler = StandardScaler()
        classifier = make_embedding_estimator(RANDOM_SEED)
        classifier.fit(scaler.fit_transform(embeddings[ann["emb_row"]]), y_labeled)
        predicted = classifier.predict(scaler.transform(embeddings))
    else:
        pipeline = make_classical_pipeline(model_name, RANDOM_SEED)
        pipeline.fit(ann["model_text"], y_labeled)
        predicted = pipeline.predict(raw["model_text"])
    predictions[f"pred_{task}"] = predicted
    labeled_pred = predictions.set_index("record_key").loc[ann["record_key"], f"pred_{task}"]
    agreement_rows.append({
        "task": task,
        "selected_model": model_name,
        "heldout_test_macro_f1": selected["tasks"][task]["test_macro_f1"],
        "refit_agreement_macro_f1": float(f1_score(y_labeled, labeled_pred, average="macro", labels=labels, zero_division=0)),
        "refit_agreement_accuracy": float(accuracy_score(y_labeled, labeled_pred)),
        "agreement_meaning": "Calculated after refitting on all 1,199 labeled rows. This is not a held-out test score.",
    })
    for scope, mask_frame, manual in (
        ("manual_annotated", ann, ann[task].astype(str)),
        ("refit_prediction_on_annotated", ann, labeled_pred),
        ("refit_prediction_on_all_comments", raw, pd.Series(predicted, index=raw.index)),
        ("refit_prediction_on_unlabeled", raw.loc[~raw["is_annotated"].astype(bool)], pd.Series(predicted, index=raw.index).loc[~raw["is_annotated"].astype(bool)]),
    ):
        counts = manual.astype(str).value_counts()
        for label, count in counts.items():
            distribution_rows.append({
                "task": task,
                "scope": scope,
                "label": label,
                "n": int(count),
                "denominator": int(len(manual)),
                "share": float(count / len(manual)),
            })

predictions.to_csv(OUT / "corpus_predictions.csv", index=False)
pd.DataFrame(distribution_rows).to_csv(OUT / "label_distribution_comparison.csv", index=False)
pd.DataFrame(agreement_rows).to_csv(OUT / "refit_agreement.csv", index=False)
print(pd.DataFrame(agreement_rows).to_string(index=False))


            task      selected_model  heldout_test_macro_f1  refit_agreement_macro_f1  refit_agreement_accuracy                                                                        agreement_meaning
policy_relevance       complement_nb               0.491360                  0.852848                  0.904921 Calculated after refitting on all 1,199 labeled rows. This is not a held-out test score.
   feedback_type logistic_regression               0.419918                  0.861615                  0.860717 Calculated after refitting on all 1,199 labeled rows. This is not a held-out test score.
           topic       complement_nb               0.410233                  0.901502                  0.884904 Calculated after refitting on all 1,199 labeled rows. This is not a held-out test score.
       sentiment       complement_nb               0.500135                  0.869360                  0.889908 Calculated after refitting on all 1,199 labeled rows. This is not a held-out test sc

### Interpretation

Each selected model is refit on all 1,199 labeled comments and then labels all 5,292 comments. On the full corpus the largest predicted classes are `relevant` (3,925), `complaint` (1,117), `corruption_unofficial_payment` (1,480), and `negative` (2,773). The refit agreement with the manual labels is high because the model has already seen those labels. The performance number remains the test macro-F1 from notebook 05.


## 3. Draw the 30-comment comparison

Sample ten annotated comments, ten unlabeled comments, and ten further comments without replacement.


In [3]:
def redact(text):
    return PHONE_LIKE.sub("[PHONE REDACTED]", str(text))[:240]

annotated_keys = ann["record_key"].tolist()
unlabeled_keys = raw.loc[~raw["is_annotated"].astype(bool), "record_key"].tolist()
generator = np.random.default_rng(RANDOM_SEED)
annotated_sample = generator.choice(annotated_keys, size=10, replace=False)
unlabeled_sample = generator.choice(unlabeled_keys, size=10, replace=False)
used = set(annotated_sample).union(unlabeled_sample)
remaining = [key for key in raw["record_key"].tolist() if key not in used]
extra_sample = generator.choice(remaining, size=10, replace=False)

frames = []
for group_name, keys in (
    ("annotated_manual_vs_predicted", annotated_sample),
    ("unlabeled_raw_with_prediction", unlabeled_sample),
    ("additional_predicted_comments", extra_sample),
):
    part = raw.loc[raw["record_key"].isin(keys), ["record_key", "post_id", "comment_original", "is_annotated"]].copy()
    part["sample_group"] = group_name
    frames.append(part)
sample = pd.concat(frames, ignore_index=True)
sample = sample.merge(predictions.drop(columns=["source_row", "post_id", "is_annotated"]), on="record_key", how="left")
manual = ann[["record_key", *CLASSIFICATION_TARGETS]].rename(columns={task: f"manual_{task}" for task in CLASSIFICATION_TARGETS})
sample = sample.merge(manual, on="record_key", how="left")
sample = sample.merge(topics[["record_key", "topic_id", "topic_terms"]], on="record_key", how="left")
sample = sample.rename(columns={"topic_id": "unsupervised_topic_id", "topic_terms": "unsupervised_topic_terms"})
sample["excerpt"] = sample["comment_original"].map(redact)
sample = sample.drop(columns=["comment_original"])
if sample["record_key"].nunique() != 30:
    raise AssertionError("The three samples are not 30 unique comments.")
if sample["excerpt"].str.contains(r"\b\d{10,15}\b", regex=True).any():
    raise AssertionError("A phone-like number remained in an excerpt.")
sample.to_csv(OUT / "comment_comparison_sample.csv", index=False)

overlap = {
    "annotated_sample": 10,
    "unlabeled_sample": 10,
    "additional_predicted_sample": 10,
    "unique_record_keys": int(sample["record_key"].nunique()),
    "overlap_between_samples": 0,
    "note": "Every comment receives a prediction. The third sample is not a separate dataset. It is ten further comments drawn from the rows that were not already sampled. The unsupervised topic columns come from notebook 04 and use a different label space from the manual topic classes.",
}
(OUT / "sample_overlap.json").write_text(json.dumps(overlap, indent=2), encoding="utf-8")
print(json.dumps(overlap, indent=2))
print("Sample groups:")
print(sample.groupby("sample_group").size().to_string())


{
  "annotated_sample": 10,
  "unlabeled_sample": 10,
  "additional_predicted_sample": 10,
  "unique_record_keys": 30,
  "overlap_between_samples": 0,
  "note": "Every comment receives a prediction. The third sample is not a separate dataset. It is ten further comments drawn from the rows that were not already sampled. The unsupervised topic columns come from notebook 04 and use a different label space from the manual topic classes."
}
Sample groups:
sample_group
additional_predicted_comments    10
annotated_manual_vs_predicted    10
unlabeled_raw_with_prediction    10


### Interpretation

The sample contains 30 unique comments and no overlap between the three draws. Eight of the ten further comments have no manual label. Where a manual label exists, four field-level differences appear in the displayed rows. A match in this sample is not a test result. Phone-like strings in the excerpts are redacted.


## How to read the comparison

Use `heldout_test_scores.csv` from notebook 05 for performance. Use `refit_agreement.csv` only as a description of the full-data refit. Use `label_distribution_comparison.csv` with the denominator in each row. A larger predicted share than the manual share can come from the model, from the class imbalance, or from a difference between the annotated rows and the other comments. It is not, by itself, evidence that the unlabeled comments have that label.
